# Ingest the documents
Source: the Delta Lake documentation (Apache-2.0), the `.md` and `.mdx` pages of `delta-io/delta` at a pinned commit, so every run sees the same corpus.

The raw files land in the volume `raw_docs`, one folder per commit, and `docs_raw` holds one row per page with its title and source URL.

In [ ]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("src_path", "")

import sys

sys.path.insert(0, dbutils.widgets.get("src_path"))
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
target = f"{catalog}.{schema}"

dbutils.widgets.text("docs_repo", "")
dbutils.widgets.text("docs_commit", "")
dbutils.widgets.text("docs_prefix", "")

docs_repo = dbutils.widgets.get("docs_repo")
docs_commit = dbutils.widgets.get("docs_commit")
docs_prefix = dbutils.widgets.get("docs_prefix")

In [ ]:
import json
import os
import urllib.request

from rag_kb.text import parse_frontmatter


def fetch(url):
    with urllib.request.urlopen(url, timeout=60) as response:
        return response.read()


tree = json.loads(fetch(f"https://api.github.com/repos/{docs_repo}/git/trees/{docs_commit}?recursive=1"))
paths = [
    item["path"] for item in tree["tree"]
    if item["path"].startswith(docs_prefix) and item["path"].endswith((".md", ".mdx"))
]
landing = f"/Volumes/{catalog}/{schema}/raw_docs/{docs_repo.replace('/', '_')}/{docs_commit}"

rows = []
for path in paths:
    local = f"{landing}/{path[len(docs_prefix):]}"
    if not os.path.exists(local):
        os.makedirs(os.path.dirname(local), exist_ok=True)
        with open(f"{local}.part", "wb") as f:
            f.write(fetch(f"https://raw.githubusercontent.com/{docs_repo}/{docs_commit}/{path}"))
        os.replace(f"{local}.part", local)
    with open(local, encoding="utf-8") as f:
        raw_text = f.read()
    meta, _ = parse_frontmatter(raw_text)
    rows.append((
        path[len(docs_prefix):], meta.get("title"), meta.get("description"), raw_text,
        f"https://github.com/{docs_repo}/blob/{docs_commit}/{path}", docs_commit,
    ))

In [ ]:
(
    spark.createDataFrame(rows, "doc_path string, title string, description string, raw_text string, source_url string, doc_commit string")
    .selectExpr("*", "current_timestamp() AS loaded_at")
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{target}.docs_raw")
)
print(f"{len(rows)} pages from {docs_repo}@{docs_commit[:7]}")